# Cross Encoder Reranking

In [ ]:
import os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sentence_transformers import CrossEncoder

# Root path setup for imports
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Loading retrieval and score computation function
from src.retrieval import rerank_with_cross_encoder

In [2]:
# Configuration
data_path = PROJECT_ROOT / "data/processed/arxiv_papers.csv"
embeddings_path = PROJECT_ROOT / "data/processed/arxiv_embeddings.npy"

data = pd.read_csv(data_path)
embeddings = np.load(embeddings_path)

In [ ]:
# Load Cross encoder
cross_encoder = CrossEncoder("cross-encoder/stsb-roberta-base")

# Example query for retrieval
query = "learning visual features without labels"

## SBERT + Cross Encoder

In [ ]:
# load SBERT model
sbert = SentenceTransformer("all-MiniLM-L6-v2")

In [5]:
# 2 Stage retrieval for SBERT
reranked_sbert = rerank_with_cross_encoder(query, data, embeddings, bi_encoder=sbert, cross_encoder=cross_encoder, k=20).sort_values("cross_score", ascending=False)

# Reranked results
reranked_sbert[['title','summary','final_score','cross_score']].head()

,title,summary,final_score,cross_score
12,Few-shot Learning for Spatial Regression,We propose a few-shot learning method for spat...,0.410880,0.669744
2,Two-Level Adversarial Visual-Semantic Coupling...,The performance of generative zero-shot method...,0.464360,0.665254
19,An empirical study of domain-agnostic semi-sup...,A class of recent semi-supervised learning (SS...,0.396743,0.659143
17,Learning Representations by Maximizing Mutual ...,We propose an approach to self-supervised repr...,0.400317,0.629328
10,Supervised Topological Maps,Controlling the internal representation space ...,0.425207,0.628559


## SPECTER + Cross Encoder

### SPECTER is trained on scientific paper semantics

In [ ]:
# Load specter model
specter = SentenceTransformer('allenai-specter')

specter_embeddings_path = PROJECT_ROOT / 'data/processed/specter_embeddings.npy'

# Generate/load embeddings
if not os.path.exists(specter_embeddings_path):
    specter_embeddings = specter.encode(data['combined'].tolist(), show_progress_bar=True)
    np.save('specter_embeddings.npy', specter_embeddings)
else:
    specter_embeddings = np.load(specter_embeddings_path)

In [ ]:
# 2 Stage retrieval for SPECTER
reranked_specter = rerank_with_cross_encoder(query, data, specter_embeddings, bi_encoder=specter, cross_encoder=cross_encoder, k=20).sort_values("cross_score", ascending=False)

# Reranked results
reranked_specter[['title','summary','final_score','cross_score']].head()

,title,summary,final_score,cross_score
14,MaxDropout: Deep Neural Network Regularization...,Different techniques have emerged in the deep ...,0.718626,0.700155
4,Sparseout: Controlling Sparsity in Deep Networks,Dropout is commonly used to help reduce overfi...,0.746162,0.685859
0,Balanced Meta-Softmax for Long-Tailed Visual R...,Deep classifiers have achieved great success i...,0.757650,0.668227
16,An empirical study of domain-agnostic semi-sup...,A class of recent semi-supervised learning (SS...,0.717857,0.659143
1,Learning Representations by Maximizing Mutual ...,We propose an approach to self-supervised repr...,0.753197,0.629328


In [ ]:
# Comparison
comparison = pd.DataFrame({
    'SBERT Top-5': reranked_sbert['title'].head(5).values,
    'SPECTER Top-5': reranked_specter['title'].head(5).values
})

with pd.option_context('display.max_colwidth', None):
    display(comparison.head())

,SBERT Top-5,SPECTER Top-5
0,Few-shot Learning for Spatial Regression,MaxDropout: Deep Neural Network Regularization Based on Maximum Output Values
1,Two-Level Adversarial Visual-Semantic Coupling for Generalized Zero-shot Learning,Sparseout: Controlling Sparsity in Deep Networks
2,An empirical study of domain-agnostic semi-supervised learning via energy-based models: joint-training and pre-training,Balanced Meta-Softmax for Long-Tailed Visual Recognition
3,Learning Representations by Maximizing Mutual Information Across Views,An empirical study of domain-agnostic semi-supervised learning via energy-based models: joint-training and pre-training
4,Supervised Topological Maps,Learning Representations by Maximizing Mutual Information Across Views
